In [55]:
import os
import certifi


from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [56]:
from langchain.agents import create_react_agent, AgentExecutor

In [58]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()



GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")     



In [59]:
search_tool = TavilySearchResults(max_results=1)

In [61]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [62]:
result = search_tool.invoke('what is today"s weather')
result

[{'url': 'https://world-weather.info/forecast/usa/new_york/may-2026/',
  'content': 'Detailed ⚡ New York City Weather Forecast for May 2026 – day/night 🌡️ ... Thursday, 14 May. Day. +61°. 2. 29.6. 70%. +55°. 05:38 AM. 08:06 PM. Waning'}]

In [63]:
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.3-70b-versatile", # This model is free, fast, and smart
    temperature=0
)

In [64]:
response = llm.invoke('When was the first icc tournament')

response 

c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:432: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response = response.dict()


AIMessage(content="The first ICC (International Cricket Council) tournament was the 1975 Cricket World Cup, which was held in England from June 7 to June 21, 1975. It was the inaugural edition of the Cricket World Cup, and it featured eight teams from around the world. The tournament was won by the West Indies, who defeated Australia in the final at Lord's Cricket Ground in London.\n\nHowever, if you're referring to the first ICC tournament in the modern sense, with the ICC as the governing body, it would be the 1987 Reliance Cup, which was held in India and Pakistan from October 8 to November 8, 1987. This tournament was the first to be organized by the ICC, which was then known as the International Cricket Conference (ICC).\n\nThe ICC was founded in 1909 as the Imperial Cricket Conference, and it was renamed the International Cricket Conference in 1965. It was renamed again to the International Cricket Council in 1989.", response_metadata={'token_usage': {'completion_tokens': 201, 'p

In [65]:
prompt = hub.pull("hwchase17/react")
prompt

c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [66]:
tools = [search_tool, get_weather_data]

In [67]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [75]:
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True   # used to se the logs of the agent
)

In [80]:
response = agent_executor.invoke({
    "input": (
        #"India's Tea garden in himachal"
        "Current weather and temperature in Palampur"
        
    )
})



> Entering new AgentExecutor chain...


c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c

Thought: To find the current weather and temperature in Palampur, I need to access a reliable source of real-time weather data. The function get_weather_data seems to be the most suitable option for this task.

Action: get_weather_data
Action Input: PalampurCould not fetch weather data for Palampur

c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c

Thought: Since the get_weather_data function was unable to fetch the weather data for Palampur, I should try to find out if Palampur is a valid city or if there are any issues with the function. However, given that Palampur is a known city in India, the issue might be with the function's ability to handle the city name or its current limitations. 

Action: tavily_search_results_json
Action Input: current weather in Palampur[{'url': 'https://www.accuweather.com/en/in/palampur/198333/may-weather/198333', 'content': "Palampur's May 2026 forecast shows daily high temperatures ranging from 74° to 101°, with overnight lows between 57° and 72°. The average high for May is"}]

c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c

Thought: I now know the final answer
Final Answer: The current weather in Palampur is not available in real-time, but the average high temperature for May 2026 is expected to range from 74° to 101°, with overnight lows between 57° and 72°.

> Finished chain.


c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\Users\Aashi\miniconda3\envs\langagent\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c

In [82]:
print(response['output'])

The current weather in Palampur is not available in real-time, but the average high temperature for May 2026 is expected to range from 74° to 101°, with overnight lows between 57° and 72°.
